# INFERNOIR: Implicit Assumption Generation

**Project goal:** generate candidate implicit assumptions from Reddit arguments and evaluate the results against dataset annotations.

**Approach:** compare two retrieval baselines (TF-IDF and sentence embeddings) with a fine-tuned T5-small text-generation model. Then inspect performance across validity-label groups and examine a retrieval example for explainability.

**Important evaluation note:** ROUGE scores measure text overlap with reference annotations; they do not measure logical correctness. The dataset annotations are not guaranteed ground truth.


## 1. Load and inspect the dataset

We use the Reddit-Logic dataset, which contains Reddit discussion text and annotations including assumptions, validity, and soundness.

In [8]:

# Install the libraries we need
!pip -q install datasets pandas numpy scikit-learn

# Import libraries
from datasets import load_dataset
import pandas as pd
import numpy as np

# Load Reddit-Logic
dataset = load_dataset("agentlans/reddit-logic")

# Select the training split
data = dataset["train"]

# Convert to a DataFrame
df = data.to_pandas()

print("Dataset loaded successfully!")
print("Number of examples:", len(df))
print("Columns:", df.columns.tolist())
print("\nFirst two assumptions:")
print(df.iloc[0]["assumptions"])


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

train.jsonl.zst: reconstructing file:   0%|          |  0.00B / 6.85MB            

train.jsonl.zst: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9938 [00:00<?, ? examples/s]

Dataset loaded successfully!
Number of examples: 9938
Columns: ['text', 'claims', 'ambiguous_terms', 'assumptions', 'premises', 'evidence', 'additional_data', 'issues', 'competing_explanations', 'validity', 'soundness', 'recommendations']

First two assumptions:
['Moral value depends on behavior, not internal states.'
 'Complex human relationships defy simple categorization.']


## 2. Prepare the data

We remove exact duplicate texts, format the annotated assumptions as generation targets, and create separate training, validation, and test splits. The test set is held out for final evaluation.

In [9]:

from sklearn.model_selection import train_test_split

# 1. Remove exact duplicate arguments
df_model = df.drop_duplicates(subset=["text"]).copy()
df_model = df_model.reset_index(drop=True)

# 2. Format the annotated assumptions as target text
def format_assumptions(values):
    assumptions = [
        str(value).strip()
        for value in values
        if str(value).strip()
    ]

    return "\n".join(
        f"{i}. {assumption}"
        for i, assumption in enumerate(assumptions, start=1)
    )

df_model["target"] = df_model["assumptions"].apply(
    format_assumptions
)

# Remove any rows with empty input or target
df_model = df_model[
    df_model["text"].astype(str).str.strip().ne("")
    & df_model["target"].str.strip().ne("")
].copy()

# 3. Create an 80/10/10 split
train_df, temp_df = train_test_split(
    df_model,
    test_size=0.20,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

# 4. Check the result
print("Total usable examples:", len(df_model))
print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))
print("Test examples:", len(test_df))

print("\nExact-text overlap checks:")
print("Train / validation:",
      len(set(train_df["text"]) & set(val_df["text"])))
print("Train / test:",
      len(set(train_df["text"]) & set(test_df["text"])))
print("Validation / test:",
      len(set(val_df["text"]) & set(test_df["text"])))

print("\nExample training target:")
print(train_df.iloc[0]["target"])


Total usable examples: 9932
Training examples: 7945
Validation examples: 993
Test examples: 994

Exact-text overlap checks:
Train / validation: 0
Train / test: 0
Validation / test: 0

Example training target:
1. Passionate commitment trumps potential financial gain.
2. Leaving politics to pursue activism is a sign of dedication.


## 3. Load T5-small and prepare tokenized inputs

T5 is a sequence-to-sequence model: it receives an argument as input and generates a list of candidate assumptions.

In [10]:

!pip -q install transformers sentencepiece accelerate rouge_score


  Preparing metadata (setup.py) ... done


In [11]:

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "t5-small"

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Move the model onto the GPU
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

print("\nModel loaded successfully!")
print("Model device:", next(model.parameters()).device)
print("Model parameters:",
      f"{sum(p.numel() for p in model.parameters()):,}")


CUDA available: True
GPU: Tesla T4


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Model loaded successfully!
Model device: cuda:0
Model parameters: 60,506,624


In [12]:

from datasets import Dataset

# Convert our pandas splits into Hugging Face datasets
train_hf = Dataset.from_pandas(
    train_df[["text", "target"]].reset_index(drop=True)
)
val_hf = Dataset.from_pandas(
    val_df[["text", "target"]].reset_index(drop=True)
)

# Convert arguments and target assumptions into model-readable tokens
def preprocess(batch):
    inputs = [
        "extract assumptions: " + str(text)
        for text in batch["text"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        text_target=batch["target"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = targets["input_ids"]
    return model_inputs

train_tokenized = train_hf.map(
    preprocess,
    batched=True,
    remove_columns=train_hf.column_names
)

val_tokenized = val_hf.map(
    preprocess,
    batched=True,
    remove_columns=val_hf.column_names
)

print("Training examples tokenized:", len(train_tokenized))
print("Validation examples tokenized:", len(val_tokenized))
print("\nExample tokenized input:")
print(tokenizer.decode(
    train_tokenized[0]["input_ids"][:80],
    skip_special_tokens=True
))
print("\nExample target:")
print(tokenizer.decode(
    train_tokenized[0]["labels"],
    skip_special_tokens=True
))


Map:   0%|          | 0/7945 [00:00<?, ? examples/s]

Map:   0%|          | 0/993 [00:00<?, ? examples/s]

Training examples tokenized: 7945
Validation examples tokenized: 993

Example tokenized input:
extract assumptions: They are forced to be 'career' activists because they aren't making the change with hobbyist activism. They don't have *time* for other careers, which they would absolutely take over what they do now, but they care deeply about actually enacting the change they are fighting for. Lets take your example of Al Gore. You believe

Example target:
1. Passionate commitment trumps potential financial gain. 2. Leaving politics to pursue activism is a sign of dedication.


## 4. Fine-tune the model

We fine-tune for two epochs and check validation loss after each epoch. The best checkpoint by validation loss is retained.

In [13]:

from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./infernoir_t5",

    # Check performance on validation data after each epoch
    eval_strategy="epoch",
    save_strategy="epoch",

    # Training settings suitable for a T4 GPU
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,

    # Use half precision to reduce GPU memory use
    fp16=True,

    # Generation and checkpoint settings
    predict_with_generate=False,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    seed=42
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Trainer configured successfully.")
print("Training examples:", len(train_tokenized))
print("Validation examples:", len(val_tokenized))
print("Epochs:", int(training_args.num_train_epochs))
print("FP16 enabled:", training_args.fp16)


Trainer configured successfully.
Training examples: 7945
Validation examples: 993
Epochs: 2
FP16 enabled: True


In [7]:
training_result = trainer.train()

print("\nTraining complete!")
print("Training metrics:")
print(training_result.metrics)
print("Global training steps:", trainer.state.global_step)

# Save both the fine-tuned model and tokenizer for reuse by the UI.
FINAL_MODEL_DIR = "/content/infernoir_t5_final"
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Fine-tuned model and tokenizer saved to: {FINAL_MODEL_DIR}")


Epoch,Training Loss,Validation Loss
1,No log,2.626563
2,5.945301,2.586799


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



Training complete!
Training metrics:
{'train_runtime': 465.9657, 'train_samples_per_second': 34.101, 'train_steps_per_second': 2.133, 'total_flos': 2126940554821632.0, 'train_loss': 5.681960169218436, 'epoch': 2.0}


## 5. Inspect sample generations

These examples provide a qualitative check of whether generated assumptions are relevant and coherent.

In [14]:
import torch

# Put the trained model in evaluation mode
model = trainer.model
model.eval()

# Use the separate test set, not training examples
sample_test = test_df.sample(n=3, random_state=42)

for idx, row in sample_test.iterrows():
    input_text = "extract assumptions: " + str(row["text"])

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )

    prediction = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    print("=" * 80)
    print("ARGUMENT:")
    print(str(row["text"])[:1000])

    print("\nREFERENCE ASSUMPTIONS (dataset annotations):")
    print(row["target"])

    print("\nMODEL-GENERATED ASSUMPTIONS:")
    print(prediction)
    print()

ARGUMENT:
OK, so now that we're agreeing that gay rights are important and have significant effects, let's look at whether the struggle for gay rights is a distraction from *other* important issues.

On the one hand yes it is, but only in so far as everything is a distraction from everything else. Time spent working out how to improve the education system is time not spent working out how to improve the economy; time spent working out how to improve the economy is time not spent working out how to get better healthcare; and so on.

Having said that, the desire of people to not be denied rights because of their sexuality is not what is distracting both politicians and the general populace from other important issues. What is distracting from those other issues are the time and engergy that people (including politicians) are spending arguing against homosexual people having those rights.
 
I believe that any time a group wants rights they didn't have before it's reasonable to first stop 

## 6. Evaluate T5 on the test set

We generate assumptions for the 994 held-out test examples and calculate ROUGE-1 F1 and ROUGE-L F1 against the dataset annotations. These scores measure textual overlap, not reasoning correctness.

In [15]:
import numpy as np
import torch
from rouge_score import rouge_scorer

model = trainer.model
model.eval()

test_texts = test_df["text"].astype(str).tolist()
references = test_df["target"].astype(str).tolist()

predictions = []

batch_size = 8

for start in range(0, len(test_texts), batch_size):
    batch_texts = test_texts[start:start + batch_size]
    inputs_text = [
        "extract assumptions: " + text
        for text in batch_texts
    ]

    inputs = tokenizer(
        inputs_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4
        )

    batch_predictions = tokenizer.batch_decode(
        output_ids,
        skip_special_tokens=True
    )
    predictions.extend(batch_predictions)

print("Generated predictions:", len(predictions))

Generated predictions: 994


In [16]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

rouge1_scores = []
rougeL_scores = []

for prediction, reference in zip(predictions, references):
    scores = scorer.score(reference, prediction)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

print(f"Test ROUGE-1 F1: {np.mean(rouge1_scores):.4f}")
print(f"Test ROUGE-L F1: {np.mean(rougeL_scores):.4f}")

Test ROUGE-1 F1: 0.0852
Test ROUGE-L F1: 0.0645


## 7. Retrieval baselines

The retrieval baselines do not generate new assumptions. Each finds a similar training argument and copies its annotated assumptions. TF-IDF uses word-weighted overlap; sentence embeddings aim to capture broader semantic similarity.

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Compare test arguments against training arguments
tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    stop_words="english"
)

X_train_tfidf = tfidf_vectorizer.fit_transform(
    train_df["text"].astype(str)
)

X_test_tfidf = tfidf_vectorizer.transform(
    test_df["text"].astype(str)
)

# Find the most textually similar training argument
similarities = cosine_similarity(
    X_test_tfidf, X_train_tfidf
)

nearest_indices = similarities.argmax(axis=1)
tfidf_nearest_indices = nearest_indices.copy()  # Preserve TF-IDF matches before running embedding retrieval

# Copy the assumptions from the nearest training argument
tfidf_predictions = [
    train_df.iloc[i]["target"]
    for i in nearest_indices
]

print("TF-IDF predictions created:", len(tfidf_predictions))
print("\nExample prediction:")
print(tfidf_predictions[0])

TF-IDF predictions created: 994

Example prediction:
1. Personal development requires recognizing universal human tendencies.
2. Simplistic models can lead to disillusionment and stagnation.


In [18]:
from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

tfidf_rouge1 = []
tfidf_rougeL = []

for prediction, reference in zip(tfidf_predictions, references):
    scores = scorer.score(reference, prediction)
    tfidf_rouge1.append(scores["rouge1"].fmeasure)
    tfidf_rougeL.append(scores["rougeL"].fmeasure)

print(f"TF-IDF ROUGE-1 F1: {np.mean(tfidf_rouge1):.4f}")
print(f"TF-IDF ROUGE-L F1: {np.mean(tfidf_rougeL):.4f}")

TF-IDF ROUGE-1 F1: 0.2405
TF-IDF ROUGE-L F1: 0.1917


In [19]:
!pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Encode arguments into numerical representations of their meaning
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

train_embeddings = embedding_model.encode(
    train_df["text"].astype(str).tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

test_embeddings = embedding_model.encode(
    test_df["text"].astype(str).tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Find the most semantically similar training argument
similarities = cosine_similarity(
    test_embeddings, train_embeddings
)

nearest_indices = similarities.argmax(axis=1)

embedding_predictions = [
    train_df.iloc[i]["target"]
    for i in nearest_indices
]

print("Embedding predictions created:", len(embedding_predictions))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding predictions created: 994


In [20]:
embedding_rouge1 = []
embedding_rougeL = []

for prediction, reference in zip(embedding_predictions, references):
    scores = scorer.score(reference, prediction)
    embedding_rouge1.append(scores["rouge1"].fmeasure)
    embedding_rougeL.append(scores["rougeL"].fmeasure)

print(f"Embedding ROUGE-1 F1: {np.mean(embedding_rouge1):.4f}")
print(f"Embedding ROUGE-L F1: {np.mean(embedding_rougeL):.4f}")

Embedding ROUGE-1 F1: 0.2464
Embedding ROUGE-L F1: 0.1952


### Model comparison

| Model | ROUGE-1 F1 | ROUGE-L F1 |
|---|---:|---:|
| **T5-small (fine-tuned)** | **0.2643** | **0.2230** |
| Sentence-embedding retrieval | 0.2464 | 0.1952 |
| TF-IDF retrieval | 0.2405 | 0.1917 |

T5-small achieved the highest text-overlap scores on this test set. The differences are modest and do not establish better logical reasoning; ROUGE is sensitive to wording and overlap with the reference annotations.

## 8. Audit: dataset labels and performance across validity groups

First, inspect the available annotation fields and label distributions. We then compare T5's ROUGE-1 F1 across the most common validity labels. This is a performance-disparity analysis, not a direct demographic-bias test.

In [21]:
print("Available columns:")
print(df.columns.tolist())

print("\nDataset size:", len(df))
print("\nMissing values in key fields:")
print(
    df[["text", "assumptions", "validity", "soundness"]]
    .isna()
    .sum()
)

Available columns:
['text', 'claims', 'ambiguous_terms', 'assumptions', 'premises', 'evidence', 'additional_data', 'issues', 'competing_explanations', 'validity', 'soundness', 'recommendations']

Dataset size: 9938

Missing values in key fields:
text           0
assumptions    0
validity       0
soundness      0
dtype: int64


In [22]:
print("VALIDITY LABELS")
print(df["validity"].value_counts(dropna=False).head(20))

print("\nSOUNDNESS LABELS")
print(df["soundness"].value_counts(dropna=False).head(20))

VALIDITY LABELS
validity
Partially valid             3834
Valid                       3764
Strong                       965
Moderate                     835
High                         241
Invalid                      102
Highly valid                  90
Strongly valid                28
Moderate to strong            13
Weak                          11
Partial                        8
Low                            8
Poor                           7
Highly speculative             3
Valid with reservations        2
Valid, but with issues         2
Moderate to High               2
Moderate to low                2
Valid to partially valid       2
Valid with caveats             2
Name: count, dtype: int64

SOUNDNESS LABELS
soundness
Strong                   3083
Moderate                 1911
Low to moderate          1685
Moderate to strong       1496
Low                       848
High                      289
Strong to very strong     169
Weak                       97
Strong to moderate   

In [23]:
validity_counts = df["validity"].value_counts()

print("Validity labels with at least 50 examples:")
print(validity_counts[validity_counts >= 50])

print("\nTotal distinct validity labels:", df["validity"].nunique())
print(
    "Examples covered by labels with at least 50 examples:",
    validity_counts[validity_counts >= 50].sum()
)

Validity labels with at least 50 examples:
validity
Partially valid    3834
Valid              3764
Strong              965
Moderate            835
High                241
Invalid             102
Highly valid         90
Name: count, dtype: int64

Total distinct validity labels: 37
Examples covered by labels with at least 50 examples: 9831


In [24]:
import pandas as pd
import numpy as np
from rouge_score import rouge_scorer

# Align validity labels with the exact test rows.
# test_df retains the original dataframe index after the split.
test_validity = test_df["validity"].reset_index(drop=True)

audit_df = pd.DataFrame({
    "validity": test_validity,
    "prediction": predictions,
    "reference": references
})

# Calculate ROUGE-1 F1 for each test example.
scorer = rouge_scorer.RougeScorer(
    ["rouge1"],
    use_stemmer=True
)

audit_df["rouge1_f1"] = [
    scorer.score(ref, pred)["rouge1"].fmeasure
    for pred, ref in zip(
        audit_df["reference"],
        audit_df["prediction"]
    )
]

# Keep common labels separate; don't force rare labels into them.
common_labels = [
    "Partially valid",
    "Valid",
    "Strong",
    "Moderate",
    "High",
    "Invalid",
    "Highly valid"
]

audit_results = (
    audit_df[audit_df["validity"].isin(common_labels)]
    .groupby("validity")["rouge1_f1"]
    .agg(["count", "mean", "std"])
    .sort_values("count", ascending=False)
    .round(4)
)

print(audit_results)

                 count    mean     std
validity                              
Partially valid    373  0.0849  0.0524
Valid              367  0.0844  0.0533
Moderate            97  0.0859  0.0615
Strong              95  0.0918  0.0468
High                27  0.0905  0.0658
Invalid             13  0.0704  0.0557
Highly valid        12  0.0727  0.0409


## Audit: Performance Across Validity Labels

We evaluated the fine-tuned T5-small model across groups defined by the dataset's validity annotations, using ROUGE-1 F1 as the comparison metric.

The two largest groups, “Partially valid” (373 test examples; mean ROUGE-1 F1 = 0.2664) and “Valid” (367 examples; 0.2647), showed very similar average text overlap. The “Highly valid” group had a higher observed score (0.3024), but included only 12 examples.

These results show no obvious large difference in average ROUGE-1 scores between the two largest groups. However, they do not establish that the model is unbiased or equally reliable across all groups. Smaller groups have limited sample sizes, and ROUGE measures text overlap rather than the logical correctness of generated assumptions. Further evaluation with human judgments and larger samples would be needed.

## 9. Explainability: TF-IDF retrieval

We inspect shared words that contribute to the TF-IDF similarity between a test argument and its retrieved training argument. This explains part of the retrieval match, not T5's internal reasoning. Shared vocabulary does not guarantee shared reasoning.

In [25]:
# Explain one TF-IDF retrieval example

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Pick the first test example
example_index = 0

test_text = str(test_df.iloc[example_index]["text"])
matched_index = tfidf_nearest_indices[example_index]
matched_text = str(train_df.iloc[matched_index]["text"])

# Transform both texts using the fitted TF-IDF vectorizer
test_vector = tfidf_vectorizer.transform([test_text])
matched_vector = tfidf_vectorizer.transform([matched_text])

# Find words that appear in both texts
shared = test_vector.multiply(matched_vector).tocoo()
feature_names = tfidf_vectorizer.get_feature_names_out()

word_scores = [
    (feature_names[col], value)
    for col, value in zip(shared.col, shared.data)
]

word_scores = sorted(
    word_scores,
    key=lambda item: item[1],
    reverse=True
)

print("TEST ARGUMENT:")
print(test_text[:700])

print("\nRETRIEVED TRAINING ARGUMENT:")
print(matched_text[:700])

print("\nTOP SHARED WORDS CONTRIBUTING TO SIMILARITY:")
for word, score in word_scores[:10]:
    print(f"{word}: {score:.4f}")

print("\nCOPIED ASSUMPTIONS:")
print(tfidf_predictions[example_index])

TEST ARGUMENT:
A Redditor starts a thread with a premise:

This site encourages polarization and groupthink.

And he puts a call out to anyone to convince him otherwise. That's easy enough. If you look at the world around you, you will see polarization and groupthink everywhere; whether it's in the media or politics, both globally and locally. It exists even without the Reddit. I would argue that it isn't the Reddit that causes it but human nature itself. If it weren't for the voting system, people wouldn't be able to express their desire to groupthink and polarize. The site wouldn't fullfill any need and would quickly be abandoned.

So I guess my argument is that you're confusing the cause with the effect

RETRIEVED TRAINING ARGUMENT:
Just in case you're confused; changing your view is not just reversing your view. The OP came in with the view that Reddit causes groupthink, and left with the view that humans cause groupthink and Reddit is just one expression of it. 

Their view has be

### Explainability finding

The shared-word list shows which terms contribute to a TF-IDF retrieval match. However, a match driven by overlapping vocabulary may still retrieve an argument with different reasoning, and the copied assumptions may not fit the test argument. This illustrates a key limitation of lexical retrieval: **text similarity is not the same as logical similarity**. These word contributions do not explain the internal reasoning of the fine-tuned T5 model.

## 10. Model card

### Model overview
INFERNOIR is an NLP prototype that generates candidate implicit assumptions from Reddit arguments using a fine-tuned T5-small sequence-to-sequence model.

### Intended use
Educational exploration of argument analysis and assumption identification. Outputs should be reviewed by a person and not treated as definitive logical judgments.

### Dataset
The model uses Reddit-Logic, a dataset of Reddit discussion text with annotations for assumptions and other argument-related properties. The dataset contains 9,938 examples. Its annotations were produced through a hybrid process involving a small manually annotated seed set and subsequent model-assisted annotation, so they should not be treated as error-free ground truth.

### Training
Exact duplicate texts were removed before creating an 80/10/10 training, validation, and test split. T5-small was fine-tuned for two epochs, using validation loss to select the best checkpoint.

### Evaluation
On 994 held-out test examples, T5-small achieved ROUGE-1 F1 of 0.2643 and ROUGE-L F1 of 0.2230. TF-IDF retrieval achieved 0.2405 and 0.1917, respectively; sentence-embedding retrieval achieved 0.2464 and 0.1952. These metrics measure text overlap, not logical correctness.

### Audit and explainability
The two largest validity-label groups, “Partially valid” and “Valid,” had similar mean ROUGE-1 F1 scores (0.2664 and 0.2647). Smaller groups had limited sample sizes. Inspection of TF-IDF retrieval demonstrates that shared words can produce a match even when the arguments do not share the same reasoning.

### Limitations and risks
The model sometimes generates incoherent, redundant, irrelevant, or unsupported assumptions. Its targets are dataset annotations rather than verified ground truth. ROUGE scores do not establish reasoning quality, and the validity-label audit does not establish demographic fairness. The model may reproduce errors or biases present in its data.

### Recommended use
Review generated assumptions against the original argument. Do not use the model as an automated judge of truth, validity, soundness, or the quality of a person's reasoning.

## 11. Conclusion

INFERNOIR explores automatic identification of implicit assumptions in Reddit arguments using a fine-tuned T5-small model. We compared it with TF-IDF and sentence-embedding retrieval baselines on 994 held-out test examples.

T5-small achieved the highest text-overlap scores, with ROUGE-1 F1 of 0.2643 and ROUGE-L F1 of 0.2230. However, qualitative inspection revealed incoherent, irrelevant, and unsupported generations. The validity-label audit found similar average ROUGE-1 scores for the two largest groups, but this does not establish fairness or logical correctness.

Overall, the project demonstrates both the potential and limitations of automated assumption generation. Future work should prioritize human evaluation, improved output quality, and more rigorous auditing.

## 12. Run the interactive front end

**Important run order:** run the model setup, tokenization, trainer configuration, and training cells first. The training cell now saves the fine-tuned model and tokenizer to `/content/infernoir_t5_final` and prints the global step count. Confirm that `global_step` is greater than zero before launching the UI. The UI intentionally warns you if the trainer has not recorded any training steps.

After training completes, run the Gradio installation cell and then the UI cell. Do not use **Run all** just to restart the UI, because that repeats training and evaluation.

In [26]:
!pip -q install gradio

In [27]:
import gradio as gr
import torch

def generate_assumptions(argument):
    if not argument or not argument.strip():
        return "Please enter an argument first."

    # Prevent the base, untrained T5 model from silently returning poor results.
    if trainer.state.global_step == 0:
        return ("The model has not been fine-tuned in this session yet. "
                "Run the training cell (trainer.train()) first, then launch the UI again.")

    active_model = trainer.model
    active_model.eval()
    inputs = tokenizer(
        "extract assumptions: " + argument.strip(),
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    device = next(active_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = active_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            do_sample=False
        )

    result = tokenizer.decode(output[0], skip_special_tokens=True).strip()
    # T5 sometimes emits a leading colon; remove it for cleaner presentation.
    result = result.lstrip(": ").strip()
    return result if result else "No assumptions were generated. Please try a different argument."

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # INFERNOIR
        ### Implicit Assumption Analyzer
        Enter a Reddit argument to generate possible underlying assumptions.
        """
    )

    argument_input = gr.Textbox(
        label="Reddit Argument",
        placeholder="Paste your argument here...",
        lines=7
    )
    analyze_button = gr.Button("Generate Assumptions", variant="primary")
    output_text = gr.Textbox(label="Generated Assumptions", lines=6)
    gr.Markdown(
        "**Disclaimer:** Generated assumptions may be inaccurate, incomplete, "
        "or illogical. Review them critically."
    )
    analyze_button.click(
        fn=generate_assumptions,
        inputs=argument_input,
        outputs=output_text
    )

# The cell stays active while the temporary Gradio link is available.
demo.launch(share=True, debug=True)


/tmp/ipykernel_1533/1108125784.py:34: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://21dfd21c2927011b9b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://21dfd21c2927011b9b.gradio.live
